In [1]:
import pandas as pd
import numpy as np
import os

print("Power BI data preparation started.")

Power BI data preparation started.


In [2]:
market_prices = pd.read_csv(
    "../data/processed/market_prices_clean.csv"
)

market_returns = pd.read_csv(
    "../data/processed/market_returns.csv"
)

portfolio_returns = pd.read_csv(
    "../data/processed/portfolio_returns_risk.csv"
)

portfolio_holdings = pd.read_csv(
    "../data/processed/portfolio_holdings.csv"
)

portfolio_summary = pd.read_csv(
    "../data/processed/portfolio_risk_summary.csv"
)

print("\nData loaded successfully.")

print("Market prices:", market_prices.shape)
print("Market returns:", market_returns.shape)
print("Portfolio returns:", portfolio_returns.shape)
print("Portfolio holdings:", portfolio_holdings.shape)
print("Portfolio summary:", portfolio_summary.shape)



Data loaded successfully.
Market prices: (11865, 8)
Market returns: (11865, 9)
Portfolio returns: (1694, 6)
Portfolio holdings: (7, 3)
Portfolio summary: (6, 2)


In [3]:
market_prices["Date"] = pd.to_datetime(
    market_prices["Date"]
)

market_returns["Date"] = pd.to_datetime(
    market_returns["Date"]
)

portfolio_returns["Date"] = pd.to_datetime(
    portfolio_returns["Date"]
)

print("\nDate columns standardized.")



Date columns standardized.


In [4]:
asset_performance = (
    market_returns
    .groupby("Ticker")["Daily_Return"]
    .agg(
        Average_Daily_Return="mean",
        Daily_Volatility="std"
    )
    .reset_index()
)

asset_performance["Annualized_Return"] = (
    asset_performance["Average_Daily_Return"] * 252
)

asset_performance["Annualized_Volatility"] = (
    asset_performance["Daily_Volatility"] * np.sqrt(252)
)

asset_performance["Return_to_Risk_Ratio"] = (
    asset_performance["Annualized_Return"]
    /
    asset_performance["Annualized_Volatility"]
)

asset_performance = asset_performance.round(6)

print("\nAsset performance calculated.")

asset_performance


Asset performance calculated.


,Ticker,Average_Daily_Return,Daily_Volatility,Annualized_Return,Annualized_Volatility,Return_to_Risk_Ratio
0,AAPL,0.001096,0.019706,0.276162,0.312831,0.882785
1,GLD,0.000643,0.011688,0.162057,0.185535,0.873458
2,JPM,0.000794,0.019234,0.200010,0.305330,0.655061
3,MSFT,0.000902,0.019126,0.227355,0.303611,0.748838
4,NVDA,0.002682,0.032592,0.675880,0.517386,1.306335
5,SPY,0.000640,0.012630,0.161278,0.200496,0.804398
6,TLT,-0.000162,0.010368,-0.040710,0.164594,-0.247338


In [5]:
asset_returns_matrix = (
    market_returns
    .pivot(
        index="Date",
        columns="Ticker",
        values="Daily_Return"
    )
    .dropna()
)

weights = (
    portfolio_holdings
    .set_index("Ticker")["Weight"]
)

covariance_matrix = asset_returns_matrix.cov()

portfolio_covariance = covariance_matrix.loc[
    weights.index,
    weights.index
]

portfolio_volatility = (
    weights.T
    @ portfolio_covariance
    @ weights
) ** 0.5

marginal_contribution = (
    portfolio_covariance @ weights
) / portfolio_volatility

risk_contribution = (
    weights * marginal_contribution
)

risk_contribution_pct = (
    risk_contribution
    / portfolio_volatility
)

risk_contribution = pd.DataFrame({
    "Ticker": weights.index,
    "Weight": weights.values,
    "Risk_Contribution": risk_contribution.values,
    "Risk_Contribution_Pct": risk_contribution_pct.values
})

risk_contribution = risk_contribution.reset_index(drop=True)

print("\nRisk contribution calculated.")

print(
    "Total risk contribution:",
    risk_contribution["Risk_Contribution_Pct"].sum()
)

risk_contribution


Risk contribution calculated.
Total risk contribution: 0.9999999999999999


,Ticker,Weight,Risk_Contribution,Risk_Contribution_Pct
0,AAPL,0.15,0.002358,0.178053
1,MSFT,0.15,0.002350,0.177480
2,JPM,0.15,0.001816,0.137112
3,NVDA,0.15,0.004103,0.309885
4,SPY,0.20,0.002366,0.178695
5,GLD,0.10,0.000261,0.019747
6,TLT,0.10,-0.000013,-0.000973


In [6]:
stress_scenarios = {

    "Market Crash": {
        "AAPL": -0.10,
        "MSFT": -0.10,
        "JPM": -0.10,
        "NVDA": -0.10,
        "SPY": -0.10,
        "GLD": 0.00,
        "TLT": 0.00
    },

    "Severe Market Crash": {
        "AAPL": -0.20,
        "MSFT": -0.20,
        "JPM": -0.20,
        "NVDA": -0.20,
        "SPY": -0.20,
        "GLD": 0.00,
        "TLT": 0.00
    },

    "Commodity Shock": {
        "AAPL": 0.00,
        "MSFT": 0.00,
        "JPM": 0.00,
        "NVDA": 0.00,
        "SPY": 0.00,
        "GLD": -0.15,
        "TLT": 0.00
    }
}


stress_results = []

for scenario, shocks in stress_scenarios.items():

    portfolio_impact = 0

    for ticker, shock in shocks.items():

        weight = weights[ticker]

        portfolio_impact += (
            weight * shock
        )

    stress_results.append({
        "Scenario": scenario,
        "Portfolio_Impact": portfolio_impact,
        "Portfolio_Impact_Pct": portfolio_impact * 100
    })


stress_tests = pd.DataFrame(
    stress_results
)

print("\nStress testing calculated.")

stress_tests


Stress testing calculated.


,Scenario,Portfolio_Impact,Portfolio_Impact_Pct
0,Market Crash,-0.080,-8.0
1,Severe Market Crash,-0.160,-16.0
2,Commodity Shock,-0.015,-1.5


In [7]:
daily_returns = (
    portfolio_returns["Portfolio_Return"]
)

annualized_return = (
    daily_returns.mean() * 252
)

annualized_volatility = (
    daily_returns.std() * np.sqrt(252)
)

risk_free_rate = 0

sharpe_ratio = (
    annualized_return - risk_free_rate
) / annualized_volatility

var_95 = (
    daily_returns.quantile(0.05)
)

expected_shortfall_95 = (
    daily_returns[
        daily_returns <= var_95
    ].mean()
)


# Cumulative return

portfolio_returns["Cumulative_Return"] = (
    1 + portfolio_returns["Portfolio_Return"]
).cumprod()


# Running peak

portfolio_returns["Running_Peak"] = (
    portfolio_returns["Cumulative_Return"]
    .cummax()
)


# Drawdown

portfolio_returns["Drawdown"] = (
    portfolio_returns["Cumulative_Return"]
    /
    portfolio_returns["Running_Peak"]
) - 1


maximum_drawdown = (
    portfolio_returns["Drawdown"].min()
)


# Latest rolling volatility

latest_rolling_volatility = (
    portfolio_returns["Rolling_30D_Volatility"]
    .dropna()
    .iloc[-1]
)


risk_metrics = pd.DataFrame({
    "Metric": [
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Historical VaR 95%",
        "Expected Shortfall 95%",
        "Maximum Drawdown",
        "Latest 30D Rolling Volatility"
    ],

    "Value": [
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        var_95,
        expected_shortfall_95,
        maximum_drawdown,
        latest_rolling_volatility
    ]
})


print("\nPortfolio risk metrics calculated.")

risk_metrics


Portfolio risk metrics calculated.


,Metric,Value
0,Annualized Return,0.251301
1,Annualized Volatility,0.210199
2,Sharpe Ratio,1.195542
3,Historical VaR 95%,-0.018691
4,Expected Shortfall 95%,-0.029821
5,Maximum Drawdown,-0.321823
6,Latest 30D Rolling Volatility,0.122921


In [8]:
portfolio_returns_powerbi = portfolio_returns[
    [
        "Date",
        "Portfolio_Return",
        "Cumulative_Return",
        "Running_Peak",
        "Drawdown",
        "Rolling_30D_Volatility"
    ]
].copy()


print("\nPortfolio returns dataset prepared.")

portfolio_returns_powerbi.head()


Portfolio returns dataset prepared.


,Date,Portfolio_Return,Cumulative_Return,Running_Peak,Drawdown,Rolling_30D_Volatility
0,2020-01-03,-0.006354,0.993646,0.993646,0.000000,NaN
1,2020-01-06,0.003337,0.996962,0.996962,0.000000,NaN
2,2020-01-07,-0.003468,0.993505,0.996962,-0.003468,NaN
3,2020-01-08,0.005908,0.999375,0.999375,0.000000,NaN
4,2020-01-09,0.008397,1.007766,1.007766,0.000000,NaN


In [9]:
market_prices_powerbi = market_prices[
    [
        "Date",
        "Ticker",
        "Adj_Close",
        "Close",
        "High",
        "Low",
        "Open",
        "Volume"
    ]
].copy()


print("\nMarket prices dataset prepared.")

market_prices_powerbi.head()


Market prices dataset prepared.


,Date,Ticker,Adj_Close,Close,High,Low,Open,Volume
0,2020-01-02,AAPL,72.271561,75.087502,75.150002,73.797501,74.059998,135480400
1,2020-01-02,GLD,143.949997,143.949997,144.210007,143.399994,143.860001,7733800
2,2020-01-02,JPM,117.899216,141.089996,141.100006,139.259995,139.789993,10803700
3,2020-01-02,MSFT,151.544250,160.619995,160.729996,158.330002,158.779999,22622100
4,2020-01-02,NVDA,5.957138,5.997750,5.997750,5.918000,5.968750,237536000


In [10]:
market_returns_powerbi = market_returns.copy()

market_returns_powerbi["Daily_Return_Pct"] = (
    market_returns_powerbi["Daily_Return"] * 100
)

print("\nAsset returns dataset prepared.")

market_returns_powerbi.head()


Asset returns dataset prepared.


,Date,Ticker,Adj_Close,Close,High,Low,Open,Volume,Daily_Return,Daily_Return_Pct
0,2020-01-02,AAPL,72.271561,75.087502,75.150002,73.797501,74.059998,135480400,NaN,NaN
1,2020-01-02,GLD,143.949997,143.949997,144.210007,143.399994,143.860001,7733800,NaN,NaN
2,2020-01-02,JPM,117.899216,141.089996,141.100006,139.259995,139.789993,10803700,NaN,NaN
3,2020-01-02,MSFT,151.544250,160.619995,160.729996,158.330002,158.779999,22622100,NaN,NaN
4,2020-01-02,NVDA,5.957138,5.997750,5.997750,5.918000,5.968750,237536000,NaN,NaN


In [11]:
portfolio_holdings_powerbi = portfolio_holdings.copy()

portfolio_holdings_powerbi["Weight_Pct"] = (
    portfolio_holdings_powerbi["Weight"] * 100
)

print("\nPortfolio holdings dataset prepared.")

portfolio_holdings_powerbi


Portfolio holdings dataset prepared.


,Ticker,Weight,Asset_Class,Weight_Pct
0,AAPL,0.15,Equity,15.0
1,MSFT,0.15,Equity,15.0
2,JPM,0.15,Equity,15.0
3,NVDA,0.15,Equity,15.0
4,SPY,0.20,Equity ETF,20.0
5,GLD,0.10,Commodity ETF,10.0
6,TLT,0.10,Bond ETF,10.0


In [12]:
dashboard_path = "../dashboard"

os.makedirs(
    dashboard_path,
    exist_ok=True
)


market_prices_powerbi.to_csv(
    f"{dashboard_path}/powerbi_market_prices.csv",
    index=False
)

market_returns_powerbi.to_csv(
    f"{dashboard_path}/powerbi_asset_returns.csv",
    index=False
)

portfolio_returns_powerbi.to_csv(
    f"{dashboard_path}/powerbi_portfolio_returns.csv",
    index=False
)

portfolio_holdings_powerbi.to_csv(
    f"{dashboard_path}/powerbi_portfolio_holdings.csv",
    index=False
)

asset_performance.to_csv(
    f"{dashboard_path}/powerbi_asset_performance.csv",
    index=False
)

risk_metrics.to_csv(
    f"{dashboard_path}/powerbi_risk_metrics.csv",
    index=False
)

risk_contribution.to_csv(
    f"{dashboard_path}/powerbi_risk_contribution.csv",
    index=False
)

stress_tests.to_csv(
    f"{dashboard_path}/powerbi_stress_tests.csv",
    index=False
)


print("\n================================================")
print("POWER BI DATA EXPORT COMPLETE")
print("================================================")

print("\nFiles created:")

for file in os.listdir(dashboard_path):
    print("-", file)


POWER BI DATA EXPORT COMPLETE

Files created:
- powerbi_asset_performance.csv
- powerbi_asset_returns.csv
- powerbi_market_prices.csv
- powerbi_portfolio_holdings.csv
- powerbi_portfolio_returns.csv
- powerbi_risk_contribution.csv
- powerbi_risk_metrics.csv
- powerbi_stress_tests.csv
